In [0]:
from pyspark.sql.functions import col , lit , trim , to_date , current_timestamp
from pyspark.sql.types import DecimalType , LongType


In [0]:
from datetime import datetime

team_name  = "team_lemma"
bronze_db  = f"charles_schwab_retailbrokerage_dev_{team_name}.bronze"
silver_db  = f"charles_schwab_retailbrokerage_dev_{team_name}.silver"
staging_db = f"charles_schwab_retailbrokerage_dev_{team_name}.staging"
catalog  = f"charles_schwab_retailbrokerage_dev_{team_name}"

spark.sql("USE CATALOG charles_schwab_retailbrokerage_dev_team_lemma")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

In [0]:
dbutils.widgets.text("batch_id" , "1" , "Batch ID 1 , 2 OR 3")

In [0]:


batch_id = dbutils.widgets.get("batch_id")

print("batch_id = ", batch_id)

In [0]:

try:
    if batch_id == "1":
        run_info_now = spark.sql(f"""SELECT _run_id , _batch FROM {bronze_db}.dailymarket 
                                WHERE _batch = '1'
                                LIMIT 1
                                """).first()
        
        carried_run_id = run_info_row[0] if run_info_row else "unknown"
        carried_batch = run_info_row[1] if run_info_row else f"Batch{batch_id}"
    else:
        run_info_now = spark.sql(f"""SELECT _run_id , _batch FROM {staging_db}.dailymarket 
                                WHERE _batch = '1'
                                LIMIT 1
                                """).first()
        
        carried_run_id = run_info_row[0] if run_info_row else "unknown"
        carried_batch = run_info_row[1] if run_info_row else f"Batch{batch_id}"
except Exception:
    carried_run_id = "unknown"
    carried_batch = f"Batch{batch_id}"

In [0]:
run_id = carried_batch

In [0]:
if batch_id == "1":
      spark.sql("CREATE SCHEMA IF NOT EXISTS sliver")

      sliver_b1 =  spark.table(f"{bronze_db}.dailymarket")\
                        .filter(col("_batch") == '1')\
                        .select(
                              to_date(trim(col("DM_DATE")),   "yyyy-MM-dd").alias("dm_date"),
                              trim(col("DM_S_SYMB")).alias("dm_s_symb"),
                              trim(col("DM_CLOSE")).cast(DecimalType(8,2)).alias("dm_close"),
                              trim(col("DM_HIGH")).cast(DecimalType(8,2)).alias("dm_high"),
                              trim(col("DM_LOW")).cast(DecimalType(8,2)).alias("dm_low"),
                              trim(col("DM_VOL")).cast(LongType()).alias("dm_vol"),
                              col("_batch"),
                              lit(carried_run_id).alias("_run_id"),
                              current_timestamp().alias("_load_ts"),
                        )
                  
      ## write in the delta table
      sliver_b1.write\
            .format("delta")\
            .mode("overwrite")\
            .saveAsTable(f"{silver_db}.markethistory") 
      
      sliver_count_b1 = spark.table(f"{silver_db}.markethistory").count()
      print(f"sliver markethistory B1 rows :{sliver_count_b1}")
      


In [0]:
from delta.tables import DeltaTable

In [0]:
if batch_id != "1":
    batch_folder = f"Batch{batch_id}"

    staged_tbl = spark.table(f"{staging_db}.dailymarket_current")\
                     .filter(col("cdc_action").isin("N", "C" , "D"))


    sliver_tbl = DeltaTable.forName(
        spark,
        f"{catalog}.silver.markethistory"

    )

    (
        sliver_tbl.alias("s")
        .merge(
            staged_tbl.alias("i"),
            "s.dm_date = TRY_TO_DATE(TRIM(i.DM_DATE), 'yyyy-MM-dd')"
             "AND s.dm_s_symb = TRIM(i.DM_S_SYMB)"
        )
        .whenMatchedUpdate(
            condition = "i.cdc_action = 'C'",
            set = {
                "dm_close": "CAST(TRIM(i.DM_CLOSE) AS DECIMAL(8,2))",
                "dm_high": "CAST(TRIM(i.DM_HIGH) AS DECIMAL(8,2))",
                "dm_low": "CAST(TRIM(i.DM_LOW) AS DECIMAL(8,2))",
                "dm_vol": "CAST(TRIM(i.DM_VOL) AS BIGINT)",
                "_batch": f"'{batch_folder}'",
                "_run_id": f"'{run_id}'",
                "_load_ts": "current_timestamp()"
            }
        )
        .whenMatchedDelete(
            condition = "i.cdc_action = 'D'"
            
        )
        .whenNotMatchedInsert(
            condition="i.cdc_action = 'N'",
            values={
                "dm_date": "TRY_TO_DATE(TRIM(i.DM_DATE), 'yyyy-MM-dd') ",
                "dm_s_symb": "TRIM(i.DM_S_SYMB)",
                "dm_close": "CAST(TRIM(i.DM_CLOSE) AS DECIMAL(8,2))",
                "dm_high": "CAST(TRIM(i.DM_HIGH) AS DECIMAL(8,2))",
                "dm_low": "CAST(TRIM(i.DM_LOW) AS DECIMAL(8,2))",
                "dm_vol": "CAST(TRIM(i.DM_VOL) AS BIGINT)",
                "_batch": f"'{batch_folder}'",
                "_run_id": f"'{run_id}'",
                "_load_ts": "current_timestamp()"
            }
        )
        .execute()
    )

    total = spark.table(f"{silver_db}.markethistory").count()
    print(f"silver markethistory rows after {batch_folder}:{total}")
       


In [0]:
print( spark.table(f"{bronze_db}.dailymarket").count() )

In [0]:

expected_count = spark.table(f"{bronze_db}.dailymarket").count()




checks =  {
    "silver.markethistory" : (spark.table(f"{silver_db}.markethistory").count(),  expected_count)
}

print(f"\n{'Table':<25} {'actual':>10} {'expected':>10} {'Status'}")
print("-"*60)
for table, (actual, expected) in checks.items():
    Status = "pass" if actual == expected else "FAIL"
    print(f"{table:<25} {actual:>10} {expected:>10} {Status}")

In [0]:
%run ../../02_common_utils/operations

In [0]:

if batch_id == "1":    
    source_count = (
        spark.table(f"{bronze_db}.dailymarket")
        .filter(col("_batch") == "1")
        .count()
    )
    target_count = (
        spark.table(f"{silver_db}.markethistory")
        .filter(col("_batch") == "1")
        .count()
    )
    carried_run_id = str(
        spark.table(f"{silver_db}.markethistory")
        .filter(col("_batch") == "1")
        .select("`_run_id`").first()[0]
    )

    log_pipeline_recon(
        spark=spark,
        run_id=carried_run_id,
        batch_id="1",
        domain="MARKET",
        table_name="markethistory",
        source_layer="bronze",
        target_layer="silver",
        source_count=source_count,
        target_count=target_count
    )

    log_audit_event(
        spark=spark,
        run_id=carried_run_id,
        batch="1",
        layer="silver",
        table_name="markethistory",
        operation="OVERWRITE",
        rows_affected=target_count
    )

    print(f"source_count : {source_count:,}")   
    print(f"target_count : {target_count:,}")   

In [0]:
if batch_id != "1":
    staging_df = spark.table(f"{staging_db}.dailymarket_current")

    source_count = staging_df.filter(col("cdc_action").isin("N", "C")).count()


    target_count = (
        spark.table(f"{silver_db}.markethistory")
        .filter(col("_batch") == f"{batch_id}")
        .count()
    )

    carried_run_id = str(staging_df.select("`_run_id`").first()[0])

    log_pipeline_recon(
        spark=spark,
        run_id=carried_run_id,
        batch_id= f"{batch_id}",
        domain="MARKET",
        table_name="markethistory",
        source_layer="staging",
        target_layer="silver",
        source_count=source_count,
        target_count=target_count
    )

    log_audit_event(
        spark=spark,
        run_id=carried_run_id,
        batch=f"{batch_id}",
        layer="silver",
        table_name="markethistory",
        operation="MERGE",
        rows_affected=source_count
    )




 


    
   